## Modeling

Input: `data/processed/X_train.parquet`, `X_test.parquet`, `y_train.parquet`, `y_test.parquet`

Mô hình: Random Forest, HistGradientBoosting

Metric ưu tiên: **Recall** của nhãn Fail và Withdrawn

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path().resolve().parent))
from config import RANDOM_SEED

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

In [ ]:
data_dir = Path('../data/processed')

X_train = pd.read_parquet(data_dir / 'X_train.parquet')
X_test  = pd.read_parquet(data_dir / 'X_test.parquet')
y_train = pd.read_parquet(data_dir / 'y_train.parquet').squeeze()
y_test  = pd.read_parquet(data_dir / 'y_test.parquet').squeeze()

# Fail=0, Pass=1, Withdrawn=2  (sorted alphabetically trong preprocessing)
LABEL_MAP   = {'Fail': 0, 'Pass': 1, 'Withdrawn': 2}
TARGET_NAMES = ['Fail', 'Pass', 'Withdrawn']

print('X_train:', X_train.shape, '| Features:', list(X_train.columns))
print('NaN in X_train:', X_train.isna().sum()[X_train.isna().sum() > 0].to_dict())

### Xử lý NaN còn lại

`submission_rate = NaN` khi `num_due = 0` (chưa có bài nào đến hạn tại mốc đó).
- **HistGB**: xử lý NaN natively → không cần làm gì.
- **RF**: không chịu NaN → fill `0` (ngữ nghĩa: chưa có bài đến hạn = rate không xác định, 0 là giá trị trung tính hợp lý vì `no_submission_despite_due=0` đã che trường hợp này).

In [ ]:
X_train_rf = X_train.fillna(0)
X_test_rf  = X_test.fillna(0)

# HistGB dùng trực tiếp X_train / X_test (giữ NaN)
print('NaN còn lại cho RF  — train:', X_train_rf.isna().sum().sum(), '| test:', X_test_rf.isna().sum().sum())
print('NaN còn lại cho HGB — train:', X_train.isna().sum().sum(),    '| test:', X_test.isna().sum().sum())

---

### Hàm đánh giá

In [ ]:
def evaluate(model_name, y_true, y_pred):
    sep = '=' * 50
    print(sep)
    print(model_name)
    print(sep)
    print(classification_report(y_true, y_pred, target_names=TARGET_NAMES, digits=3))

    cm = confusion_matrix(y_true, y_pred)
    fig, ax = plt.subplots(figsize=(5, 4))
    ConfusionMatrixDisplay(cm, display_labels=TARGET_NAMES).plot(ax=ax, colorbar=False)
    ax.set_title(model_name)
    plt.tight_layout()
    plt.show()

---

### Random Forest

In [ ]:
rf = RandomForestClassifier(
    n_estimators=300,
    class_weight='balanced',
    random_state=RANDOM_SEED,
    n_jobs=-1,
)
rf.fit(X_train_rf, y_train)
y_pred_rf = rf.predict(X_test_rf)

evaluate('Random Forest', y_test, y_pred_rf)

---

### HistGradientBoosting

In [ ]:
hgb = HistGradientBoostingClassifier(
    max_iter=300,
    class_weight='balanced',
    random_state=RANDOM_SEED,
)
hgb.fit(X_train, y_train)
y_pred_hgb = hgb.predict(X_test)

evaluate('HistGradientBoosting', y_test, y_pred_hgb)

---

### So sánh Recall — Fail & Withdrawn

In [ ]:
from sklearn.metrics import recall_score

results = []
for name, y_pred in [('Random Forest', y_pred_rf), ('HistGB', y_pred_hgb)]:
    recalls = recall_score(y_test, y_pred, average=None, labels=[0, 1, 2])
    results.append({
        'Model':     name,
        'Recall Fail':      round(recalls[0], 3),
        'Recall Pass':      round(recalls[1], 3),
        'Recall Withdrawn': round(recalls[2], 3),
        'Macro Recall':     round(recalls.mean(), 3),
    })

pd.DataFrame(results).set_index('Model')

---

### Feature Importance

In [ ]:
def plot_importance(model, feature_names, title, top_n=14):
    importances = pd.Series(model.feature_importances_, index=feature_names)
    importances = importances.sort_values(ascending=True).tail(top_n)

    fig, ax = plt.subplots(figsize=(7, 5))
    importances.plot(kind='barh', ax=ax)
    ax.set_title(title)
    ax.set_xlabel('Importance')
    plt.tight_layout()
    plt.show()


plot_importance(rf,  X_train_rf.columns, 'Feature Importance — Random Forest')
plot_importance(hgb, X_train.columns,    'Feature Importance — HistGB')

---

### Bayesian Optimization — HistGradientBoosting

Dùng **Optuna** tối ưu hoá macro recall của Fail và Withdrawn (bỏ qua Pass vì đã cao sẵn).  
Validation bằng 3-fold cross-validation trên tập train để tránh overfit vào test set.

In [ ]:
import optuna
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import make_scorer, recall_score

optuna.logging.set_verbosity(optuna.logging.WARNING)

# Metric: mean recall của Fail (0) và Withdrawn (2) — bỏ qua Pass
def fail_withdrawn_recall(y_true, y_pred):
    recalls = recall_score(y_true, y_pred, average=None, labels=[0, 1, 2], zero_division=0)
    return (recalls[0] + recalls[2]) / 2

scorer = make_scorer(fail_withdrawn_recall)
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_SEED)


def objective(trial):
    params = {
        'max_iter':          trial.suggest_int('max_iter', 200, 600),
        'learning_rate':     trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'max_depth':         trial.suggest_int('max_depth', 3, 10),
        'min_samples_leaf':  trial.suggest_int('min_samples_leaf', 10, 100),
        'l2_regularization': trial.suggest_float('l2_regularization', 1e-4, 10.0, log=True),
        'max_leaf_nodes':    trial.suggest_int('max_leaf_nodes', 15, 63),
        'class_weight':      'balanced',
        'random_state':      RANDOM_SEED,
    }
    model = HistGradientBoostingClassifier(**params)
    scores = cross_val_score(model, X_train, y_train, cv=cv, scoring=scorer, n_jobs=-1)
    return scores.mean()


study = optuna.create_study(direction='maximize',
                            sampler=optuna.samplers.TPESampler(seed=RANDOM_SEED))
study.optimize(objective, n_trials=50, show_progress_bar=True)

print('Best CV score (mean Fail+Withdrawn recall):', round(study.best_value, 4))
print('Best params:', study.best_params)

In [ ]:
hgb_tuned = HistGradientBoostingClassifier(
    **study.best_params,
    class_weight='balanced',
    random_state=RANDOM_SEED,
)
hgb_tuned.fit(X_train, y_train)
y_pred_hgb_tuned = hgb_tuned.predict(X_test)

evaluate('HistGB (Tuned)', y_test, y_pred_hgb_tuned)

In [ ]:
# So sánh baseline vs tuned
results_all = []
for name, y_pred in [('RF (baseline)', y_pred_rf),
                     ('HistGB (baseline)', y_pred_hgb),
                     ('HistGB (tuned)', y_pred_hgb_tuned)]:
    recalls = recall_score(y_test, y_pred, average=None, labels=[0, 1, 2])
    results_all.append({
        'Model':            name,
        'Recall Fail':      round(recalls[0], 3),
        'Recall Pass':      round(recalls[1], 3),
        'Recall Withdrawn': round(recalls[2], 3),
        'Macro Recall':     round(recalls.mean(), 3),
    })

pd.DataFrame(results_all).set_index('Model')